In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
import spacy
from emoatlas import EmoScores
from emoatlas.resources import _valences
from scipy import stats
from tqdm import tqdm
from collections import Counter
import json
import networkx as nx
import re
import os
import gc  # For garbage collection
import psutil  # For memory monitoring
from SpreadPy.Models.models import BaseSpreading
from SpreadPy import IOUtils, AnalyticalUtils
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler, Normalizer
import sys
import seaborn as sns
# It is recommended to use the new seaborn objects interface
import seaborn.objects as so

# Add project root (parent of this scripts/ folder) to sys.path once
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Now import from ../src folder (qui ho creato delle funzioni utili)
from src import exporter, fmtools, fmn_valence

In [ ]:
# you do not need to run these download commands every time. nltk.download()
# saves these data packages to your local file system in the nltk_data directory.
# Once they are downloaded, they are available for NLTK to use in any session.

'''nltk.download('wordnet')
nltk.download('omw-1.4') # for italian and multilingual support, if needed
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng') # for word tokenization
nltk.download('averaged_perceptron_tagger') # for POS tagging
'''


In [ ]:
from nltk import word_tokenize, pos_tag

# Environment and models settings

In [ ]:
# Set language once; everything else adapts
language = "italian"  # "english" or "italian"
env_file_path = "../env"

def load_env_vars(path: str) -> None:
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                s = line.strip()
                if s and not s.startswith("#"):
                    key, value = s.split("=", 1)
                    os.environ[key] = value
        print(f"Loaded SPACY_MODEL_PATH: {os.environ.get('SPACY_MODEL_PATH')}")
        print(f"Loaded NLTK_DATA: {os.environ.get('NLTK_DATA')}\n")
    else:
        print(f"Warning: {path} file not found. Using default paths.")

def ensure_nltk_path():
    nltk_data_dir = os.environ.get("NLTK_DATA")
    if nltk_data_dir:
        try:
            import nltk
            if nltk_data_dir not in nltk.data.path:
                nltk.data.path.insert(0, nltk_data_dir)
        except Exception as e:
            print(f"Warning: could not set NLTK_DATA path: {e}")

def init_spacy_and_emo(language: str):
    SPACY_DEFAULTS = {
        "english": "en_core_web_lg",
        "italian": "it_core_news_lg",
    }
    # env overrides default
    requested = os.environ.get("SPACY_MODEL_PATH") or SPACY_DEFAULTS.get(language, "en_core_web_lg")

    def _load(model_name_or_path: str):
        try:
            return spacy.load(model_name_or_path), model_name_or_path
        except Exception as e:
            return None, e

    nlp, ok = _load(requested)
    if nlp is None:
        fallback = SPACY_DEFAULTS.get(language, "en_core_web_lg")
        if isinstance(ok, Exception):
            print(f"spaCy load failed for '{requested}': {ok}\nFalling back to '{fallback}'")
        nlp, ok2 = _load(fallback)
        if nlp is None:
            raise RuntimeError(
                f"Failed to load spaCy model. Tried '{requested}' and fallback '{fallback}'. "
                f"Install with: python -m spacy download {fallback}"
            )
        requested = fallback

    emo = EmoScores(language=language, spacy_model=requested)
    return nlp, emo

# Execute setup
load_env_vars(env_file_path)
ensure_nltk_path()

nlp, emo = init_spacy_and_emo(language)

In [ ]:
pd.set_option('display.max_rows', 100)

# fmn settings

In [ ]:
# Dynamic output paths system for pipelines
base_output_path = '../out'
project_name = 'merged'
formatted_data_dir = 'formatting'
# Set the type of fmn to use: 'bfmns' or 'tfmns'
# (this is used for the folder name, not for the file names)
fmn_type = 'tfmns'
# Keywords that we want to check for in the textual data. If not empty,
# semantic frame plots will be created for these keywords. Otherwise,
# the full FMNs will be visualized.
keywords = []
keywords = ['matematica', 'ansia', 'stress', 'depressione']

# Types of output: aggregated and individual (entire dataset vs per participant FMNs)
# Choose any subset: {'aggregated'}, {'individual'}, or both
# How to use:
# Only aggregated: methods = {'aggregated'}
# Only individual: methods = {'individual'}
# Both: methods = {'aggregated', 'individual'}
ALLOWED_METHODS = {'aggregated', 'individual'}
methods = {'aggregated'}
methods = {m.strip().lower() for m in (methods if isinstance(methods, (list, set, tuple)) else [methods])}
methods = methods & ALLOWED_METHODS
if not methods:
    raise ValueError("No valid methods selected. Use 'aggregated' and/or 'individual'.")

# Groups to subset the data (if needed). Use labels and .startswith() condition
# from the 'participant_id' column to create groups.
group_name = 'physics'  # fmn name when using groups
if group_name:
    fmn_type = f'{group_name}_{fmn_type}'

participant_prefixes = ['fisica']

# If dfs_and_group_labels is longer than 1, merge the data from the different paths.
dfs_and_group_labels = []
df1 = pd.read_csv('../data/extracted_data_2025/free_text.csv')
dfs_and_group_labels = [
    (df1, participant_prefixes),
]

# Replace cells containing only a period with NaN
for df, _ in dfs_and_group_labels:
    df['text'] = df['text'].replace(r'^\s*\.\s*$', np.nan, regex=True)

# print all unique participants
all_participants = set()
for df, group_labels in dfs_and_group_labels:
    all_participants.update(df['participant_id'].unique())
print(f"All unique participants across datasets: {len(all_participants)}")
print(f"Unique participants labels: {all_participants}\n")

# Resolve output directories
if methods == {'aggregated'}:
    files_output_dir = f'{base_output_path}/{project_name}/aggregated/{fmn_type}'
    data_output_dir = f'{base_output_path}/{project_name}/aggregated/{formatted_data_dir}/{fmn_type}'
if methods == {'individual'}:
    files_output_dir = f'{base_output_path}/{project_name}/individual/{fmn_type}'
    data_output_dir = f'{base_output_path}/{project_name}/individual/{formatted_data_dir}/{fmn_type}'

### Data pre-processing functions

In [ ]:
def get_wordnet_pos(treebank_tag):
    """Convert POS tag to a format recognized by WordNetLemmatizer."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Default to noun

# Lemmatization function for English text using NLTK
def lemmatize_text(text):
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [
        lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags
    ]
    return ' '.join(lemmatized_tokens)

# Lemmatization for Italian text using spaCy (NLTK lemmatization is in English)
def lemmatize_text_it(text):
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc])

## Dataframe preparation

In [ ]:
def _normalize_token(x: str):
    '''Normalize a token by lowercasing and removing punctuation/numbers.
    It’s used in build_association_df to normalize cue and association fields.

    Parameters
    ----------
    x : str
        The input token to normalize.

    Returns
    -------
    str or None
        The normalized token, or None if the input is None or results in an empty string.
    '''
    if x is None or pd.isna(x):
        return None
    x = str(x).lower()
    x = re.sub(r'[\d\W_]+', ' ', x)      # rimuovi punteggiatura/numeri
    x = re.sub(r'\s+', ' ', x).strip()
    return x if x else None


def build_association_df(df: pd.DataFrame) -> pd.DataFrame:
    '''Convert a wide-format DataFrame of cue and association words into a
    long-format DataFrame. Doesn't create records for None/empty words, but
    doesn't drop the cue word if it's valid but has no valence value.
    '''
    assoc_pattern = re.compile(r"^association_(\d+)$")
    association_numbers = sorted({
        int(m.group(1)) for col in df.columns for m in [assoc_pattern.match(col)] if m
    })

    # Prepare records (list of dictionaries) for long format.
    # Each record corresponds to a participant's cue or association word.
    # The cue word is always included as index 0.
    # Association words are included with their respective indices.
    records = []
    for _, row in df.iterrows():
        participant_id = row.get("participant_id")
        cue_raw = row.get("cue_word")
        norm_cue = _normalize_token(cue_raw)

        # Include the cue word if it and its valence are valid.
        # Cue word is associated to itself with index 0 to allow keeping
        # its valence value in the long format.
        if norm_cue is not None:
            val_cue = pd.to_numeric(row.get("valence_cue_word"), errors="coerce")
            # Add the cue word entry to records
            records.append({
                "participant_id": participant_id,
                "cue_word": norm_cue,
                "index": 0,
                "word": norm_cue,
                "valence": val_cue
            })

        # Process association words
        for n in association_numbers:
            word_col = f"association_{n}"
            val_col = f"valence_association_{n}"
            if word_col not in df.columns:
                continue
            raw_word = row.get(word_col)
            norm_word = _normalize_token(raw_word)
            # Skip if the normalized word is None or empty
            if norm_word is None:
                continue
            val_raw = row.get(val_col) if val_col in df.columns else None
            numeric_valence = pd.to_numeric(val_raw, errors="coerce")
            # Only add if the normalized word is valid.
            # No records are created for None/empty words.
            records.append({
                "participant_id": participant_id,
                "cue_word": norm_cue,
                "index": n,
                "word": norm_word,
                "valence": numeric_valence
            })

    # Convert records to DataFrame
    long_df = pd.DataFrame.from_records(records)
    # Ensure correct types for index and valence
    if not long_df.empty:
        long_df["index"] = long_df["index"].astype(int)
        long_df["valence"] = pd.to_numeric(long_df["valence"], errors="coerce")
    return long_df


def subset_by_group_labels(dfs_and_group_labels, col: str) -> pd.DataFrame:
    """Subset and merge DataFrames by group labels.
    Accepts an iterable of (df, group_labels) pairs.
    group_labels can be a list/tuple of prefixes or None/[] to include all rows.
    Prints unique counts before/after per dataset and total after merging.

    Parameters:
    dfs_and_group_labels : iterable of (pd.DataFrame, list/tuple/None)
        An iterable where each element is a tuple containing a DataFrame and
        an optional list or tuple of group label prefixes to filter by.
        If group_labels is None or empty, no filtering is applied to that DataFrame.
    col : str
        The column name on which to perform the uniqueness checks and filtering.
    
    Returns:
    pd.DataFrame
        The merged DataFrame after subsetting by group labels.
    """
    if not dfs_and_group_labels:
        raise ValueError("The input iterable is empty.")

    # Validate same columns across all DataFrames
    first_df, _ = dfs_and_group_labels[0]
    base_columns = set(first_df.columns)
    if col not in base_columns:
        raise ValueError(f"Column '{col}' not found in DataFrames.")
    for df, _ in dfs_and_group_labels[1:]:
        if set(df.columns) != base_columns:
            raise ValueError("All DataFrames must have the same columns.")

    # Subset each DataFrame with reporting
    subset_dfs = []
    for i, (df, group_labels) in enumerate(dfs_and_group_labels, start=1):
        before_unique = df[col].astype("string").nunique(dropna=True)

        if group_labels:
            subset = df[df[col].astype("string").str.startswith(tuple(group_labels), na=False)].copy()
            filter_info = f"labels={list(group_labels)}"
        else:
            subset = df.copy()
            filter_info = "no filtering"

        after_unique = subset[col].astype("string").nunique(dropna=True)
        removed = before_unique - after_unique
        print(f"[{i}] unique '{col}' before: {before_unique} | after: {after_unique} | removed: {removed}")

        # Print the filtered labels from each dataframe
        print(f"[{i}] filtered rows from '{col}':")
        if group_labels:
            filtered_labels = subset[col].astype("string").unique()
            for label in filtered_labels:
                print(f"    {label}")

        subset_dfs.append(subset)

    # Merge subsets and report total
    merged_df = pd.concat(subset_dfs, ignore_index=True)
    total_after = merged_df[col].astype("string").nunique(dropna=True)
    print(f"[total] unique '{col}' after subsetting and merging: {total_after}")

    return merged_df

## Forma Mentis Networks functions

In [ ]:
def monitor_memory():
    """Monitor current memory usage"""
    process = psutil.Process(os.getpid())
    memory_mb = process.memory_info().rss / 1024 / 1024
    print(f"Current memory usage: {memory_mb:.2f} MB")
    return memory_mb


def group_texts(df, col, labels, exclude_pattern=None, preprocess_txt=None):
    """Group texts by labels from the grouping column (col).

    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame containing the texts.
    col : str
        The name of the column containing the texts data.
    labels : str or list[str]
        The row labels to apply for grouping texts (supports 'startswith').
    exclude_pattern : str | list[str], optional
        Regex pattern(s) removed from each grouped text before preprocessing.
    preprocess_txt : callable(str) -> str, optional
        A function applied to each grouped text AFTER exclude_pattern removal.
        Use this to plug in lowercasing, spelling fixes, lemmatization, etc.

    Returns
    -------
    tuple[dict, str]
        (dict[label] = processed text, concatenated_all_texts)
    """
    texts_by_label = {}
    all_texts = ""

    for label_str in labels:
        mask = df[col].str.startswith(label_str)
        sliced_texts = "".join(df[mask]['text'].fillna(''))

        # Remove instruction / noise patterns first
        if exclude_pattern is not None:
            if isinstance(exclude_pattern, str):
                sliced_texts = re.sub(pattern=exclude_pattern, repl='', string=sliced_texts)
            else:
                for pattern in exclude_pattern:
                    sliced_texts = re.sub(pattern=pattern, repl='', string=sliced_texts)

        # Optional custom preprocessing
        if preprocess_txt is not None:
            try:
                out = preprocess_txt(sliced_texts)
                if isinstance(out, str) and out:
                    sliced_texts = out
                elif out is None:
                    # Leave original text unchanged
                    pass
                else:
                    print(f"[group_texts] preprocess returned non-string for {label_str}, keeping original.")
            except Exception as e:
                print(f"[group_texts] preprocess failed for {label_str}: {e}")

        texts_by_label[label_str] = sliced_texts
        all_texts += sliced_texts

    return texts_by_label, all_texts


def build_tfmns_by_label(texts_by_label, emo=emo, keywords=None, labels=None, batch_size=10):
    '''
    Builds forma mentis networks for given keywords and grouped texts (by label).
    Process in batches to manage memory usage and avoid memory leaks.
    The FMNs are stored in a dictionary where keys are the fmnt names
    ("keyword_label" or just "label" if keywords is None) and values are the
    corresponding FMN objects (instances of the `FormaMentisNetwork` class).

    Args:
        texts_by_label (dict): A dictionary with labels as keys and texts as values.
        emo (EmoScores): An initialized EmoScores object.
        keywords (list): A list of target words for the networks. Defaults to None.
        labels (list): Specific labels to process. Defaults to None.
        batch_size (int): Number of networks to process in each batch.
    
    Returns:
        dict: A dictionary of the generated forma mentis networks (excluding empty networks).
    '''
    # Initialize a dictionary to store all forma mentis networks
    all_fmnts = {}
    
    # If labels is provided, filter the texts_by_label dictionary
    if labels is not None:
        texts_by_label = {label: texts for label, texts in texts_by_label.items() if label in labels}
    
    # Convert to list for batch processing
    items = list(texts_by_label.items())
    total_items = len(items)
    
    print(f"Processing {total_items} items in batches of {batch_size}")
    monitor_memory()
    
    # Process in batches
    for i in range(0, total_items, batch_size):
        batch_items = items[i:i+batch_size]
        batch_fmnts = {}
        
        print(f"\nProcessing batch {i//batch_size + 1}/{(total_items + batch_size - 1)//batch_size}")
        
        # If keywords are provided, build FMNs for each provided keyword
        # and label
        if keywords is not None:
            for keyword in keywords:
                for label, texts in batch_items:
                    print(f"Processing {keyword} for {label}")
                    fmnt_title = f"{keyword}_{label}"
                    
                    try:
                        # Build the textual forma mentis network
                        fmnt = emo.formamentis_network(text=texts, target_word=keyword, semantic_enrichment=['synonyms'], multiplex=False)
                        
                        # Check if the network has vertices before adding it
                        if hasattr(fmnt, "vertices") and len(fmnt.vertices) > 0:
                            batch_fmnts[fmnt_title] = fmnt
                        else:
                            print(f"Skipping {fmnt_title}: network has zero vertices.")
                            # Remove participant from participant_ids
                            participant_ids.discard(label)

                    except ValueError as e:
                        if "min() arg is an empty sequence" in str(e):
                            print(f"Skipping {fmnt_title}: not enough data to build the network.")
                        else:
                            raise
        else:
            for label, texts in batch_items:
                print(f"Processing {label}")
                fmnt_title = f"{label}"
                
                try:
                    # Build the textual forma mentis network
                    fmnt = emo.formamentis_network(text=texts, target_word=None, semantic_enrichment=['synonyms'], multiplex=False)
                    
                    # Check if the network has vertices before adding it
                    if hasattr(fmnt, "vertices") and len(fmnt.vertices) > 0:
                        batch_fmnts[fmnt_title] = fmnt
                    else:
                        print(f"Skipping {fmnt_title}: network has zero vertices.")
                        
                except ValueError as e:
                    if "min() arg is an empty sequence" in str(e):
                        print(f"Skipping {fmnt_title}: not enough data to build the network.")
                    else:
                        raise
        
        # Add batch to main dictionary
        all_fmnts.update(batch_fmnts)
        
        # Memory management
        monitor_memory()
        gc.collect()
    
    print(f"Successfully built {len(all_fmnts)} networks with vertices > 0")
    return all_fmnts


# An updated version of emo.export_whole_fmnt() that supports custom valence exports.
def export_whole_fmn_custom(fmnt, save_path, custom_valences, language="english"):
    """
    Export a FMN with optional custom valence node lists.
    """
    # Collect edges robustly (handles dict-of-layers or flat iterables)
    def _collect_edges(f):
        edges_attr = getattr(f, "edges", None)
        out = []
        if edges_attr is None:
            return out
        if isinstance(edges_attr, dict):
            for lst in edges_attr.values():
                for e in lst:
                    if isinstance(e, (list, tuple)) and len(e) >= 2:
                        out.append((e[0], e[1]))
        else:
            for e in edges_attr:
                if isinstance(e, (list, tuple)) and len(e) >= 2:
                    out.append((e[0], e[1]))
        return out

    edges = _collect_edges(fmnt)

    # Include all nodes: vertices union edge endpoints
    all_nodes = set(getattr(fmnt, "vertices", []) or [])
    for u, v in edges:
        all_nodes.update([u, v])

    # Build node lists
    if custom_valences:
        if not isinstance(custom_valences, (list, tuple)) or len(custom_valences) != 3:
            raise ValueError("custom_valences must be [positive_set, negative_set, neutral_set]")
        pos_set, neg_set, neu_set = custom_valences
        if not all(isinstance(s, set) for s in (pos_set, neg_set, neu_set)):
            raise ValueError("custom_valences elements must be sets")
        positive_nodes = sorted([n for n in all_nodes if n in pos_set])
        negative_nodes = sorted([n for n in all_nodes if n in neg_set])
        neutral_nodes  = sorted([n for n in all_nodes if (n in neu_set) or (n not in pos_set and n not in neg_set)])
    else:
        pos_lex, neg_lex, _ = _valences(language)
        positive_nodes = sorted([n for n in all_nodes if n in pos_lex])
        negative_nodes = sorted([n for n in all_nodes if n in neg_lex])
        neutral_nodes  = sorted([n for n in all_nodes if n not in pos_lex and n not in neg_lex])

    # Write file
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        f.write("# Forma Mentis Network Export\n\n")
        f.write("## Edges\n")
        for n1, n2 in edges:
            f.write(f"{n1} -- {n2}\n")

        f.write("\n## Positive Nodes\n")
        for node in positive_nodes:
            f.write(f"{node}\n")

        f.write("\n## Negative Nodes\n")
        for node in negative_nodes:
            f.write(f"{node}\n")

        f.write("\n## Neutral Nodes\n")
        for node in neutral_nodes:
            f.write(f"{node}\n")

    print(f"[✔] Network exported successfully to '{save_path}'")


def export_whole_fmn_multiple_custom(fmnts, save_path, filename, custom_valences, language="english"):
    """
    Export multiple FMNs with (optional) custom valences into txt files.
    Accepts dict or list. Produces unique filenames for lists.
    """
    if not os.path.exists(save_path):
        os.makedirs(save_path, exist_ok=True)

    if isinstance(fmnts, list):
        if len(fmnts) == 1:
            out_path = os.path.join(save_path, f"{filename}.txt")
            export_whole_fmn_custom(fmnts[0], out_path, custom_valences, language=language)
        else:
            for i, item in enumerate(fmnts, start=1):
                out_path = os.path.join(save_path, f"{filename}_{i}.txt")
                export_whole_fmn_custom(item, out_path, custom_valences, language=language)

    elif isinstance(fmnts, dict):
        for name, fmn in fmnts.items():
            out_path = os.path.join(save_path, f"{name}.txt")
            export_whole_fmn_custom(fmn, out_path, custom_valences, language=language)


def make_emo_flowers(fmnts: dict, save_path: str, filename: str, emo=emo, batch_size=10, svg: bool = False):
    """
    Generates emotional flowers for each forma mentis network found in fmnts.
    Saves images to save_path without displaying them.
    """
    out_dir = save_path
    os.makedirs(out_dir, exist_ok=True)

    items = list(fmnts.items())
    total_items = len(items)
    print(f"Processing {total_items} emotional flowers in batches of {batch_size}")

    for i in range(0, total_items, batch_size):
        batch_items = items[i:i+batch_size]
        print(f"\nProcessing flower batch {i//batch_size + 1}/{(total_items + batch_size - 1)//batch_size}")
        monitor_memory()

        for fmn_title, fmnt in batch_items:
            if not hasattr(fmnt, "vertices") or len(fmnt.vertices) == 0:
                print(f"Skipping {fmn_title}: empty network.")
                continue
            try:
                z_scores = emo.zscores(fmnt)
                emo.draw_plutchik(z_scores, reject_range=(-1.96, 1.96))

                base = filename if filename else fmn_title
                png_path = os.path.join(out_dir, f"{base}.png")
                fig = plt.gcf()
                fig.savefig(png_path, dpi=300)
                print(f"Saved {png_path}")

                if svg:
                    svg_path = os.path.join(out_dir, f"{base}.svg")
                    fig.savefig(svg_path)
                    print(f"Saved {svg_path}")
            except Exception as e:
                print(f"Error processing flower for {fmn_title}: {e}")
            finally:
                plt.close('all')
        gc.collect()


def all_fmns_features_df(features_dict: dict, orient: str = "participants",
                         count_lists: bool = False,
                         list_suffix: str = "_count",
                         drop_nested_dicts: bool = False) -> pd.DataFrame:
    """
    Build a DataFrame from the features_dict returned by extract_fmn_structural_features.

    Parameters
    ----------
    features_dict : dict
        {participant_title: {feature_key: value}}
    orient : str
        "participants" -> rows=participants, columns=features (requested)
        "features"     -> rows=features, columns=participants (original behavior)
    count_lists : bool
        If True store length of list features (e.g. positive_nodes_count).
    list_suffix : str
        Suffix appended when counting list features.
    drop_nested_dicts : bool
        If True skip nested dicts (e.g. layer_metrics). Set False to flatten one level.
    """
    def _safe_col(name):
        # keep it readable but safe for CSV headers
        return str(name).replace(" ", "_")

    rows = []
    for participant, feat_dict in features_dict.items():
        row = {"participant_id": participant}
        for k, v in feat_dict.items():
            # Special flattening
            if k == "local_clustering" and isinstance(v, dict):
                for node, coeff in v.items():
                    row[f"local_clustering.{_safe_col(node)}"] = coeff
                continue
            if k == "zscores" and isinstance(v, dict):
                for emo_k, z in v.items():
                    row[f"zscores.{_safe_col(emo_k)}"] = z
                continue
            if k == "top_5_hubs_by_degree" and isinstance(v, list):
                for i, item in enumerate(v, start=1):
                    if isinstance(item, dict):
                        row[f"hub_{i}_node"] = item.get("node")
                        row[f"hub_{i}_degree"] = item.get("degree")
                    elif isinstance(item, (list, tuple)) and len(item) >= 2:
                        row[f"hub_{i}_node"] = item[0]
                        row[f"hub_{i}_degree"] = item[1]
                # If fewer than 5 hubs, ensure consistent columns
                for i in range(len(v) + 1, 6):
                    row.setdefault(f"hub_{i}_node", None)
                    row.setdefault(f"hub_{i}_degree", None)
                continue

            # Generic handling
            if isinstance(v, dict):
                if drop_nested_dicts:
                    continue
                for sk, sv in v.items():
                    if isinstance(sv, (list, dict)):
                        continue
                    row[f"{k}.{_safe_col(sk)}"] = sv
            elif isinstance(v, list):
                if count_lists:
                    row[f"{k}{list_suffix}"] = len(v)
                else:
                    # Store as JSON-like string to keep content
                    try:
                        row[k] = json.dumps(v, ensure_ascii=False)
                    except Exception:
                        row[k] = str(v)
            else:
                row[k] = v
        rows.append(row)

    if orient == "participants":
        df = pd.DataFrame(rows)
        if not df.empty:
            df = df.set_index("participant_id").sort_index()
        return df

    # Original wide style (features as rows)
    series_list = []
    for r in rows:
        participant_id = r.pop("participant_id")
        series_list.append(pd.Series(r, name=participant_id))
    if not series_list:
        return pd.DataFrame()
    wide_df = pd.concat(series_list, axis=1)
    return wide_df


def extract_tfm_structural_features(
    fmnts: dict,
    save_path: str,
    emo=emo,
    batch_size: int = 10,
    compute_layer_metrics: bool = True,
    multiplex_merge_strategy: str = "union",
    include_zscores: bool = True,
    persist: bool = True
):
    """
    Extract structural features for Textual Forma Mentis Networks.
    Supports single-layer and multiplex FMNs (syntactic / synonyms / hypernyms).
    Produces (optionally) per-layer metrics plus merged (union) metrics.

    Parameters
    ----------
    fmnts : dict
        {title: FormamentisNetwork}
    save_path : str
        Base output directory.
    emo : EmoScores
        Initialized EmoScores instance.
    batch_size : int
        Networks per batch.
    compute_layer_metrics : bool
        If True and fmnt is multiplex (edges is a dict), compute metrics per layer.
    multiplex_merge_strategy : str
        'union' (currently implemented) – all edges merged ignoring type.
        (Placeholder for future strategies like 'intersection'.)
    include_zscores : bool
        If True compute emo.zscores(fmnt).
    persist : bool
        If True write JSON + TXT per network.

    Returns
    -------
    dict
        {fmnt_title: feature_dict}
    """
    if not os.path.exists(save_path):
        os.makedirs(save_path, exist_ok=True)

    pos_set, neg_set, _ = _valences(emo.language)

    def _sanitize_number(x):
        try:
            if x is None:
                return None
            if isinstance(x, (int, float)):
                if isinstance(x, float) and (np.isnan(x) or np.isinf(x)):
                    return None
                return float(x)
        except Exception:
            return None
        return None

    def _valence_cat(node: str) -> int:
        '''1=positive, -1=negative, 0=neutral
        
        Uses lexicon sets from _valences(emo.language).
        '''
        if node in pos_set: return 1
        if node in neg_set: return -1
        return 0

    def _valence_label(vcat: int) -> str:
        if vcat == 1: return "positive"
        if vcat == -1: return "negative"
        return "neutral"

    def _make_nx_from_edges(edgelist):
        G = nx.Graph()
        for e in edgelist:
            if len(e) >= 2:
                u, v = e[0], e[1]
                G.add_edge(u, v)
        # Add valence category and label attributes to nodes
        for n in G.nodes():
            vcat = _valence_cat(n)
            G.nodes[n]["valence_cat"] = vcat
            G.nodes[n]["valence"] = _valence_label(vcat)
        return G

    def _compute_graph_metrics(G: nx.Graph):
        n_nodes = G.number_of_nodes()
        n_edges = G.number_of_edges()
        n_pos = sum(1 for _, d in G.nodes(data=True) if d.get("valence_cat") == 1)
        n_neg = sum(1 for _, d in G.nodes(data=True) if d.get("valence_cat") == -1)

        # Degree stats
        degrees = [deg for _, deg in G.degree()]
        min_degree = min(degrees) if degrees else 0
        max_degree = max(degrees) if degrees else 0

        # Local graph-level average clustering.
        try:
            # Try the standard method first (more accurate)
            avg_clust = nx.average_clustering(G) if n_nodes > 1 else 0.0
        except Exception:
            try:
                # Fallback to approximation (less accurate but more robust)
                avg_clust = nx.algorithms.approximation.clustering_coefficient.average_clustering(G)
            except Exception:
                avg_clust = None

        # Degree assortativity
        try:
            degree_assortativity = nx.degree_assortativity_coefficient(G)
        except Exception:
            degree_assortativity = None

        # Valence assortativity
        try:
            valence_assortativity = nx.attribute_assortativity_coefficient(G, "valence")
        except Exception:
            valence_assortativity = None

        # Closeness extremes
        try:
            closeness = nx.closeness_centrality(G) if n_nodes > 0 else {}
            max_closeness = max(closeness.values()) if closeness else None
            min_closeness = min(closeness.values()) if closeness else None
        except Exception:
            max_closeness = None
            min_closeness = None

        # LCC + path stats
        if n_nodes == 0:
            lcc_aspl = None
            lcc_diameter = None
            lcc_nodes = 0
            lcc_edges = 0
        elif n_nodes == 1:
            lcc_aspl = 0.0
            lcc_diameter = 0
            lcc_nodes = 1
            lcc_edges = 0
        else:
            try:
                undirected_G = G.to_undirected() if G.is_directed() else G
                if nx.is_connected(undirected_G):
                    H = undirected_G
                else:
                    # Extract largest connected component (a set of nodes)
                    largest_cc = max(nx.connected_components(undirected_G), key=len)
                    # Create a subgraph view of the largest connected component.
                    # The graph structure cannot be changed but node/edge attributes
                    # can and are shared with the original graph.
                    H = undirected_G.subgraph(largest_cc).copy()
                lcc_nodes = H.number_of_nodes()
                lcc_edges = H.number_of_edges()
                lcc_aspl = nx.average_shortest_path_length(H) if lcc_nodes > 1 else 0.0
                lcc_diameter = nx.diameter(H) if lcc_nodes > 1 else 0
            except Exception:
                lcc_aspl = None
                lcc_diameter = None
                lcc_nodes = H.number_of_nodes() if 'H' in locals() else 0
                lcc_edges = H.number_of_edges() if 'H' in locals() else 0

            # Compute z-scores using emo module
            try:
                zscores = emo.zscores(fmnt)
            except Exception:
                zscores = {}

            # Compute local clustering
            try:
                local_clustering = nx.clustering(G)
            except Exception:
                local_clustering = {}

            # Compute hubs: top 5 concepts by highest degree
            try:
                # DegreeView is iterable of (node, degree) pairs; convert/sort directly
                degree_pairs = list(G.degree())
                sorted_degrees = sorted(degree_pairs, key=lambda x: x[1], reverse=True)[:5]
            except Exception:
                sorted_degrees = []

        return {
            "n_nodes": n_nodes,
            "n_edges": n_edges,
            "n_positive_nodes": n_pos,
            "n_negative_nodes": n_neg,
            "min_degree": _sanitize_number(min_degree),
            "max_degree": _sanitize_number(max_degree),
            "average_clustering": _sanitize_number(avg_clust),
            "local_clustering": {k: _sanitize_number(v) for k, v in local_clustering.items()},
            "degree_assortativity": _sanitize_number(degree_assortativity),
            "valence_assortativity": _sanitize_number(valence_assortativity),
            "max_closeness_centrality": _sanitize_number(max_closeness),
            "min_closeness_centrality": _sanitize_number(min_closeness),
            "lcc_nodes": _sanitize_number(lcc_nodes),
            "lcc_edges": _sanitize_number(lcc_edges),
            "lcc_average_shortest_path_length": _sanitize_number(lcc_aspl),
            "lcc_diameter": _sanitize_number(lcc_diameter),
            "zscores": {k: _sanitize_number(v) for k, v in zscores.items()},
            "top_5_hubs_by_degree": [{"node": n, "degree": _sanitize_number(d)} for n, d in sorted_degrees],
        }

    items = list(fmnts.items())
    total = len(items)
    print(f"Extracting TFMN structural features from {total} networks in batches of {batch_size}")
    monitor_memory()

    all_features = {}

    # Process in batches
    for i in range(0, total, batch_size):
        batch = items[i:i+batch_size]
        print(f"\nBatch {i//batch_size + 1}/{(total + batch_size - 1)//batch_size}")
        
        for title, fmnt in batch:
            if not hasattr(fmnt, "vertices") or len(fmnt.vertices) == 0:
                print(f"Skipping {title}: empty network.")
                continue

            # Check if the network is multiplex
            is_multiplex = isinstance(fmnt.edges, dict)
            feature_record = {
                "title": title,
                "is_multiplex": bool(is_multiplex),
            }

            # Per-layer metrics (if multiplex and requested)
            layer_metrics = {}
            layer_edge_counts = {}
            n_layers = 1

            if is_multiplex:
                # fmnt.edges is assumed to be a mapping (likely dict-like) whose keys
                # denote layers (e.g., syntactic, synonims etc...). Calling fmnt.edges.keys()
                # returns a dynamic view object;
                # wrapping it with the built-in list constructor immediately copies
                # those keys into a concrete list.
                layers = list(fmnt.edges.keys())
                n_layers = len(layers)
                feature_record["layers"] = layers
                if compute_layer_metrics:
                    for layer in layers:
                        G_layer = _make_nx_from_edges(fmnt.edges[layer])
                        layer_metrics[layer] = _compute_graph_metrics(G_layer)
                        layer_edge_counts[layer] = G_layer.number_of_edges()
                # Merge strategy
                if multiplex_merge_strategy == "union":
                    merged_edges = []
                    for layer in layers:
                        merged_edges.extend(fmnt.edges[layer])
                else:
                    # Fallback to union if unsupported strategy
                    merged_edges = []
                    for layer in layers:
                        merged_edges.extend(fmnt.edges[layer])
                G_merged = _make_nx_from_edges(merged_edges)
                merged_metrics = _compute_graph_metrics(G_merged)
                feature_record["merged_strategy"] = multiplex_merge_strategy
                for k, v in merged_metrics.items():
                    feature_record[k] = v
            else:
                # Single-layer network
                G_single = _make_nx_from_edges(fmnt.edges)
                single_metrics = _compute_graph_metrics(G_single)
                for k, v in single_metrics.items():
                    feature_record[k] = v

            feature_record["n_layers"] = n_layers
            feature_record["layer_edge_counts"] = layer_edge_counts if layer_edge_counts else None

            # Global z-scores (EmoAtlas already handles multiplex internally if passed fmnt)
            if include_zscores:
                try:
                    z_scores = emo.zscores(fmnt)
                except Exception:
                    z_scores = {}
                feature_record.update({f"z_{k}": v for k, v in z_scores.items()})

            # Node valence lists (lexicon-based)
            pos_nodes = sorted([n for n in fmnt.vertices if n in pos_set])
            neg_nodes = sorted([n for n in fmnt.vertices if n in neg_set])
            neu_nodes = sorted([n for n in fmnt.vertices if (n not in pos_set and n not in neg_set)])
            feature_record["positive_nodes"] = pos_nodes
            feature_record["negative_nodes"] = neg_nodes
            feature_record["neutral_nodes"] = neu_nodes

            # Include compact per-layer metrics if computed
            if layer_metrics:
                feature_record["layer_metrics"] = layer_metrics

            all_features[title] = feature_record

            if persist:
                json_path = os.path.join(save_path, f"{title}_tfmn_structural_features.json")
                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(feature_record, f, ensure_ascii=False, indent=2)
                txt_path = os.path.join(save_path, f"{title}_tfmn_structural_features.txt")
                with open(txt_path, "w", encoding="utf-8") as f:
                    for k, v in feature_record.items():
                        if k in ("positive_nodes", "negative_nodes", "neutral_nodes") and isinstance(v, list):
                            f.write(f"{k} ({len(v)}): {', '.join(v)}\n")
                        else:
                            f.write(f"{k}: {v}\n")
                print(f"Saved {json_path}")

        monitor_memory()
        gc.collect()

    print(f"Saved structural features for {len(all_features)} TFMNs")
    return all_features


def export_fmnts(
    fmnts: dict,
    output_subdir: str,
    output_dir: str,
    fmn_type: str = "TFMN",
    include_valences: bool = True,
    separate_layers_files: bool = False,
    merge_multiplex: bool = False,
    language: str = None,
    overwrite: bool = True
):
    """
    Unified exporter for a dictionary of Formamentis Networks.

    Handles:
      - Flat edges: [(u,v)] or typed edges [(u,v,type)]
      - Multiplex edges: {'syntactic':[(u,v)], 'synonyms':[(u,v)], ...}

    Modes for multiplex:
      separate_layers_files=True  -> writes one file per layer
      separate_layers_files=False & merge_multiplex=True  -> single merged edge list (duplicates collapsed)
      separate_layers_files=False & merge_multiplex=False -> single file with sections per layer

    Filenames:
      base: <output_dir>/<output_subdir>/<network_title>_<fmn_type>.txt
      layer split: <network_title>_<layer>_<fmn_type>.txt

    Parameters
    ----------
    fmnts : dict[str, FormamentisNetwork]
    output_subdir : str
    output_dir : str
    fmn_type : str
    include_valences : bool
    separate_layers_files : bool
    merge_multiplex : bool
    language : str or None (defaults to emo.language if available)
    overwrite : bool
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    df_output_dir = os.path.join(output_dir, output_subdir)
    os.makedirs(df_output_dir, exist_ok=True)

    # Try to infer language
    if language is None:
        try:
            # assume same EmoScores instance used globally
            lang_guess = emo.language
        except Exception:
            lang_guess = "italian"
    else:
        lang_guess = language

    pos_set, neg_set, _ = _valences(lang_guess)

    def node_val(n):
        if n in pos_set: return 'positive'
        if n in neg_set: return 'negative'
        return 'neutral'

    def write_valence_sections(fh, nodes):
        if not include_valences:
            return
        pos_nodes = sorted([n for n in nodes if n in pos_set])
        neg_nodes = sorted([n for n in nodes if n in neg_set])
        neu_nodes = sorted([n for n in nodes if n not in pos_set and n not in neg_set])
        fh.write("\n## Positive Nodes\n")
        for n in pos_nodes: fh.write(f"{n}\n")
        fh.write("\n## Negative Nodes\n")
        for n in neg_nodes: fh.write(f"{n}\n")
        fh.write("\n## Neutral Nodes\n")
        for n in neu_nodes: fh.write(f"{n}\n")

    def ensure_path(path):
        if (not overwrite) and os.path.exists(path):
            base, ext = os.path.splitext(path)
            k = 1
            new_path = f"{base}_{k}{ext}"
            while os.path.exists(new_path):
                k += 1
                new_path = f"{base}_{k}{ext}"
            return new_path
        return path

    for title, fmnt in fmnts.items():
        edges_obj = fmnt.edges
        # Multiplex case
        if isinstance(edges_obj, dict):
            if separate_layers_files:
                for layer, edgelist in edges_obj.items():
                    out_file = os.path.join(df_output_dir, f"{title}_{layer}_{fmn_type}.txt")
                    out_file = ensure_path(out_file)
                    nodes = set()
                    for u, v in edgelist:
                        nodes.update([u, v])
                    with open(out_file, "w", encoding="utf-8") as f:
                        f.write("# Forma Mentis Network Export (Layer)\n")
                        f.write(f"# Layer: {layer}\n\n")
                        f.write("## Edges\n")
                        for u, v in edgelist:
                            f.write(f"{u} -- {v}\n")
                        write_valence_sections(f, nodes)
            else:
                # Single file strategy
                out_file = os.path.join(df_output_dir, f"{title}_{fmn_type}.txt")
                out_file = ensure_path(out_file)
                if merge_multiplex:
                    merged = {}
                    for layer, edgelist in edges_obj.items():
                        for u, v in edgelist:
                            key = tuple(sorted((u, v)))
                            merged.setdefault(key, set()).add(layer)
                    all_nodes = set([n for e in merged for n in e])
                    with open(out_file, "w", encoding="utf-8") as f:
                        f.write("# Forma Mentis Network Export (Merged Multiplex)\n\n")
                        f.write("## Edges (u -- v) [layers]\n")
                        for (u, v), layers in sorted(merged.items()):
                            layers_str = ",".join(sorted(layers))
                            f.write(f"{u} -- {v} [{layers_str}]\n")
                        write_valence_sections(f, all_nodes)
                else:
                    # Sections per layer
                    all_nodes = set()
                    with open(out_file, "w", encoding="utf-8") as f:
                        f.write("# Forma Mentis Network Export (Layered Single File)\n\n")
                        for layer, edgelist in edges_obj.items():
                            f.write(f"## Layer: {layer}\n")
                            for u, v in edgelist:
                                f.write(f"{u} -- {v}\n")
                                all_nodes.update([u, v])
                            f.write("\n")
                        write_valence_sections(f, all_nodes)
        else:
            # Flat list (possibly typed)
            out_file = os.path.join(df_output_dir, f"{title}_{fmn_type}.txt")
            out_file = ensure_path(out_file)
            nodes = set()
            with open(out_file, "w", encoding="utf-8") as f:
                f.write("# Forma Mentis Network Export\n\n")
                f.write("## Edges\n")
                for e in edges_obj:
                    if len(e) >= 2:
                        u, v = e[0], e[1]
                        nodes.update([u, v])
                        if len(e) == 3:  # typed edge
                            etype = e[2]
                            f.write(f"{u} -- {v} [{etype}]\n")
                        else:
                            f.write(f"{u} -- {v}\n")
                write_valence_sections(f, nodes)


def inspect_fmn(fmn, participant = str, all_features = dict, emo=emo):
    '''Inspection of a single FMN (also multi-layer). Converts to NetworkX, prints sizes and basic checks.
       Works whether fmnt.edges is a dict (e.g., {'syntactic': [...], 'synonyms': [...]})
       or a flat iterable of 2-tuples.

    Parameters:
        fmn : FormaMentisNetwork (also multi-layer)
        participant : str
            Participant identifier for logging.
        all_features : dict
            {participant: features_dict} for printing features.
        emo : EmoScores
    '''
    def _collect_edges(edges_attr):
        # Normalize edges from different fmnt formats into a list of (u, v)
        out = []
        if edges_attr is None:
            return out
        if isinstance(edges_attr, dict):
            for lst in edges_attr.values():
                for e in lst:
                    if isinstance(e, (list, tuple)) and len(e) >= 2:
                        out.append((e[0], e[1]))
        else:
            for e in edges_attr:
                if isinstance(e, (list, tuple)) and len(e) >= 2:
                    out.append((e[0], e[1]))
        return out

    print(f"FMN: {participant}")

    # Before conversion: get vertices/edges if available
    n_vertices = len(getattr(fmn, "vertices", [])) if hasattr(fmn, "vertices") else None
    fmnt_edges = _collect_edges(getattr(fmn, "edges", None))
    fmnt_edges = list(set(tuple(map(str, e)) for e in fmnt_edges))  # dedup; ensure strings
    print("Before conversion to nx graph:")
    if n_vertices is not None:
        print(f"{n_vertices} nodes and {len(fmnt_edges)} edges")
    else:
        print(f"(vertices not available) edges: {len(fmnt_edges)}")

    # Convert to NetworkX
    fmn_G = emo.formamentis_to_nxgraph(fmn)
    print("\nAfter conversion to nx graph:")
    print(fmn_G)

    # Basic consistency check (only if we could read vertices)
    if n_vertices is not None and fmn_G.number_of_nodes() != n_vertices:
        print(f"Warning: Node count mismatch after conversion! for {participant}")

    # Compare edges (robust to undirected orientation)
    missing_edges = []
    if fmnt_edges:
        G_edges = set()
        if isinstance(fmn_G, (nx.Graph, nx.DiGraph)):
            if fmn_G.is_directed():
                G_edges = set((str(u), str(v)) for u, v in fmn_G.edges())
            else:
                G_edges = set(tuple(sorted((str(u), str(v)))) for u, v in fmn_G.edges())
        for u, v in fmnt_edges:
            e = (str(u), str(v))
            if fmn_G.is_directed():
                if e not in G_edges:
                    missing_edges.append((u, v))
            else:
                if tuple(sorted(e)) not in G_edges:
                    missing_edges.append((u, v))
    if missing_edges:
        print(f"Warning: {len(missing_edges)} edges from FMN not found in NetworkX graph for {participant}: {missing_edges[:20]}{' ...' if len(missing_edges) > 20 else ''}")

    # Print provided features (if any)
    print("\n--------------------------------------------")
    print(f"Features for {participant}:")
    print("--------------------------------------------")
    features_for_participant = (all_features or {}).get(participant, {})
    if isinstance(features_for_participant, dict):
        for k, v in features_for_participant.items():
            print(k, v)
    else:
        print(features_for_participant)
    print("\n---\n")


def draw_fmnts(
    fmnts,
    out_dir,
    layout,
    highlight,
    thickness,
    ax=None,
    emo=emo,
    custom_valences=None,
    filename_base=None,
    batch_size=5,
    figsize=(10, 10),
    dpi=300,
    bbox_pad=0.08,
    margin=0.05,
    fix_center_mask=True,
    bring_edges_to_front=True,
):
    """
    Draw FMNs and save images directly in out_dir (no subfolders).
    If filename_base is given, it is used; otherwise <fmnt_title>.

    Notes
    - For edge_bundling/circle layout, EmoAtlas draws a central white patch to hide the hub.
      Set fix_center_mask=True to make that patch transparent and avoid "missing" edges near the center.
    - bbox_inches='tight' + pad_inches keeps labels/curves fully in-frame.
    """
    import os
    import gc
    from matplotlib.patches import Circle

    os.makedirs(out_dir, exist_ok=True)

    items = list(fmnts.items())
    total_items = len(items)
    print(f"Drawing {total_items} networks in batches of {batch_size}")

    for i in range(0, total_items, batch_size):
        batch_items = items[i:i + batch_size]
        print(f"\nDrawing batch {i // batch_size + 1}/{(total_items + batch_size - 1) // batch_size}")
        monitor_memory()

        for fmnt_title, fmnt in batch_items:
            if not hasattr(fmnt, "vertices") or len(fmnt.vertices) == 0:
                print(f"Skipping {fmnt_title}: the network has zero vertices.")
                continue

            print(f"Drawing TFMN: {fmnt_title}")

            # Create a figure only if caller didn't pass one
            created_fig = False
            if ax is None:
                fig, ax_ = plt.subplots(figsize=figsize, dpi=dpi, constrained_layout=True)
                created_fig = True
            else:
                ax_ = ax
                fig = ax_.figure

            # Draw with EmoAtlas
            emo.draw_formamentis(
                fmn=fmnt,
                layout=layout,
                highlight=highlight,
                thickness=thickness,
                ax=ax_,
                save_path=None,  # prevent internal saving
                custom_valences=custom_valences,
            )

            # Optional: neutralize the central white mask that hides bundled arcs
            if fix_center_mask or bring_edges_to_front:
                try:
                    # Find smallest circle patch (the hub used by the circle layout)
                    circles = [p for p in ax_.patches if isinstance(p, Circle) and hasattr(p, "radius")]
                    if fix_center_mask and circles:
                        hub = min(circles, key=lambda c: getattr(c, "radius", float("inf")))
                        # Make it fully transparent
                        hub.set_facecolor((1, 1, 1, 0))
                        hub.set_edgecolor((1, 1, 1, 0))
                        hub.set_alpha(0.0)
                        hub.set_visible(True)
                        # Push it to the back
                        hub.set_zorder(0)

                    if bring_edges_to_front:
                        # Lines and curves live in collections
                        for coll in ax_.collections:
                            coll.set_zorder(20)
                        # Keep labels on top
                        for txt in ax_.texts:
                            txt.set_zorder(30)
                        # Other patches (outer rings, etc.) under the edges
                        for p in ax_.patches:
                            if not isinstance(p, Circle):
                                p.set_zorder(10)
                except Exception as e:
                    print(f"[WARN] Could not adjust center mask/zorder: {e}")

            # Ensure full content fits
            ax_.set_aspect("equal", adjustable="datalim")
            ax_.relim()
            ax_.autoscale_view()
            ax_.margins(x=margin, y=margin)
            ax_.axis("off")

            base = filename_base if filename_base else fmnt_title
            png_path = os.path.join(out_dir, f"{base}.png")
            svg_path = os.path.join(out_dir, f"{base}.svg")

            # Save tightly with padding to include outer labels/curves
            fig.savefig(png_path, dpi=dpi, bbox_inches="tight", pad_inches=bbox_pad, facecolor="white")
            fig.savefig(svg_path, bbox_inches="tight", pad_inches=bbox_pad, facecolor="white")
            print(f"Saved {png_path}")
            print(f"Saved {svg_path}")

            if created_fig:
                plt.close(fig)

        gc.collect()

### Memory Optimization Tips

The following optimizations have been implemented to prevent RAM issues:

1. **Batch Processing**: FMNs are built in small batches (5-10 at a time) with garbage collection between batches
2. **Memory Monitoring**: `monitor_memory()` function tracks RAM usage throughout the pipeline
3. **Optional Visualization**: Drawing and emotional flowers are commented out by default - uncomment only if needed
4. **Cleanup**: Matplotlib figures are properly closed after each use
5. **Reduced Batch Sizes**: Smaller batch sizes for memory-intensive operations

To enable visualizations, uncomment the relevant sections but be aware of memory usage.

### Data loading and pre-processing

In [ ]:
# Subset by group labels if multiple dataframes and labels provided, otherwise just merge):
final_df = subset_by_group_labels(dfs_and_group_labels, col='participant_id')
# Replace cells containing only a period with NaN
final_df['text'] = final_df['text'].replace(r'^\s*\.\s*$', np.nan, regex=True)

# Group all texts from the final dataframe into a unique string
all_texts = ' '.join(final_df['text'].dropna().tolist())
# Print to check  correct subsetting and merging
print(f"\n{all_texts}")

In [ ]:
# Export long and categorized dataframes (edit path as needed)
# Also dynamically name files based if method is aggregated or individual
if len(dfs_and_group_labels) == 1:
    exporter.export_df(final_df, data_output_dir, filename=f'{group_name}_texts')

elif len(dfs_and_group_labels) > 1:
    exporter.export_df(final_df, data_output_dir, filename=f'{group_name}_texts')

## Run the pipeline on preprocessed data

### Aggregated pipeline

In [ ]:
# Method to execute: a single aggregated FMN from all data
if 'aggregated' in methods:
    # Monitor initial memory usage
    print("Starting aggregated pipeline...\n")

    # build the aggregated FMN and print it
    all_texts_fmn = emo.formamentis_network(text=all_texts, target_word=None, semantic_enrichment=['synonyms'], multiplex=False)
        
    monitor_memory()

    export_whole_fmn_multiple_custom(
    fmnts=[all_texts_fmn],
    save_path=files_output_dir,
    filename=f'{group_name}',
    custom_valences=None,
    language=emo.language
    )

    monitor_memory()

    # Draw and save semantic frame plots and emo flowers for each keyword
    # for each keyword in keywords extract it, then draw the semantic
    # frame plot, save it and show it in the notebook
    layout = "edge_bundling"
    if len(keywords) > 0:
        for keyword in keywords:
            semantic_frame = emo.extract_word_from_formamentis(all_texts_fmn, keyword)
            if not hasattr(semantic_frame, "vertices") or len(semantic_frame.vertices) == 0:
                print(f"Skipping '{keyword}': not in network or empty semantic frame.")
                continue
            out_dir = f'{base_output_path}/{files_output_dir}/visualizations'
            filename = f'tfmn_{group_name}_{keyword}_{layout}_semantic_frame'
            draw_fmnts(
                fmnts={group_name: semantic_frame},
                out_dir=out_dir,
                layout=layout,
                highlight=[],
                thickness=1,
                ax=None,
                custom_valences=None,
                filename_base=filename,
            )
            make_emo_flowers(
                fmnts={f'{group_name}_{keyword}': semantic_frame},
                save_path=out_dir,
                filename=f'{group_name}_{keyword}_emo_flower',
                emo=emo,
                batch_size=1
            )

        monitor_memory()

    '''valence_stats = fmn_valence.fmn_valence_stats(merged_fmn, custom_valences)
    print("Valence stats:", valence_stats)

    # Inspect the merged FMN
    inspect_fmn(
        fmn=all_texts_fmn,
        participant=group_name,
        all_features={group_name: valence_stats},
        emo=emo
    )'''

    # Check for presence of each keyword in the fmn
    fmtools.verify_and_report_nodes(
        fmns={group_name: all_texts_fmn},
        required=keywords
    )
        
    # Compute and export structural features for the aggregated FMN
    aggregated_features = extract_tfm_structural_features(
        fmnts={group_name: all_texts_fmn},
        save_path=f'{base_output_path}/{files_output_dir}/structural_features',
        emo=emo,
        batch_size=1,
        persist=True
    )

    # Convert features to DataFrame and export
    aggregated_features_df = all_fmns_features_df(aggregated_features, orient="participants")
    exporter.export_df(aggregated_features_df, f'{base_output_path}/{files_output_dir}/structural_features', filename=f'{group_name}_structural_features')


### Individual pipeline

In [ ]:
# Method to execute: a single aggregated FMN from all data
if 'individual' in methods:
    print("Starting individual pipeline...\n")

    merged_fmn = emo.formamentis_network(text=all_texts, target_word=None, semantic_enrichment=['synonyms'], multiplex=False)

    tfmns_by_label = build_tfmns_by_label(
        texts_by_label=texts_by_participant,
        emo=emo,
        keywords=keywords,
        labels=list(participant_ids),
        batch_size=10
    )

    # Single flat output folder for all images
    vis_out_dir = f'{base_output_path}/{files_output_dir}/visualizations'
    os.makedirs(vis_out_dir, exist_ok=True)

    monitor_memory()

    for tfmn_title, tfmn in tfmns_by_label.items():
        # Export networks (also switch to the "individual" output root)
        export_whole_fmn_multiple_custom(
            fmnts=[tfmn],
            save_path=f'{base_output_path}/{files_output_dir}',
            filename=f'{group_name}_{tfmn_title}',
            custom_valences=None,
            language=emo.language
        )

        monitor_memory()

        layout = "edge_bundling"
        if len(keywords) > 0:
            for keyword in keywords:
                semantic_frame = emo.extract_word_from_formamentis(tfmn, keyword)
                # Save directly in the shared visualizations folder (no subfolders)
                draw_fmnts(
                    fmnts={tfmn_title: semantic_frame},
                    out_dir=vis_out_dir,
                    layout=layout,
                    highlight=[],
                    thickness=1,
                    ax=None,
                    custom_valences=None,
                    filename_base=f'{tfmn_title}_{keyword}_{layout}_semantic_frame',
                )
        else:
            # Full TFMN image, flat in the same folder
            draw_fmnts(
                fmnts={tfmn_title: tfmn},
                out_dir=vis_out_dir,
                layout=layout,
                highlight=[],
                thickness=1,
                ax=None,
                custom_valences=None,
                filename_base=f'{tfmn_title}_{layout}',
            )

        monitor_memory()


        '''valence_stats = fmn_valence.fmn_valence_stats(merged_fmn, custom_valences)
        print("Valence stats:", valence_stats)

        # Inspect the merged FMN
        inspect_fmn(
            fmn=tfmn,
            participant=group_name,
            all_features={group_name: valence_stats},
            emo=emo
        )'''

        # Check for presence of each keyword in the fmn
        fmtools.verify_and_report_nodes(
            fmns={group_name: tfmn},
            required=keywords
        )

    # Features -> also use the "individual" data folder (no per-title dirs)
    all_features = extract_tfm_structural_features(
        fmnts={tfmn_title: tfmn},
        save_path=f'{base_output_path}/{files_output_dir}/{tfmn_title}_structural_features',
        emo=emo,
    )